# Purchase Propensity Prediction Model

This notebook trains and scores customer purchase propensity models using Gold layer data.

## Overview
1. Load and explore Gold tables
2. Feature engineering
3. Model training (Logistic Regression, Random Forest, GBT)
4. MLflow experiment tracking
5. Model scoring and delta table output


In [ ]:
# Install dependencies
%pip install mlflow scikit-learn

In [ ]:
import mlflow
import mlflow.spark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")

## 1. Load Gold Tables

In [ ]:
catalog = "retails_sales_dev"

fct_vendas = spark.table(f"{catalog}.gold.fct_vendas")
dim_cliente = spark.table(f"{catalog}.gold.dim_cliente")
dim_produto = spark.table(f"{catalog}.gold.dim_produto")
dim_localidade = spark.table(f"{catalog}.gold.dim_localidade")
dim_tempo = spark.table(f"{catalog}.gold.dim_tempo")

display(fct_vendas.limit(10))

## 2. Feature Engineering

In [ ]:
# Max date for recency calculation
max_date = fct_vendas.agg(F.max("sk_data")).collect()[0][0]
print(f"Max date: {max_date}")

# Customer-level aggregations
customer_features = fct_vendas.groupBy("sk_cliente").agg(
    F.count("*").alias("total_purchases"),
    F.sum("receita").alias("total_revenue"),
    F.sum("quantidade").alias("total_quantity"),
    F.avg("receita").alias("avg_ticket"),
    F.countDistinct("sk_produto").alias("unique_products"),
    F.max("sk_data").alias("last_purchase_date"),
    F.min("sk_data").alias("first_purchase_date"),
    F.avg("margem_percentual").alias("avg_margin"),
    F.sum("lucro").alias("total_profit"),
)
    .withColumn("recency_days", F.datediff(F.lit(max_date), F.col("last_purchase_date")))
    .withColumn("customer_lifetime_days", F.datediff(F.col("last_purchase_date"), F.col("first_purchase_date")))
    .withColumn("avg_monthly_purchases", F.when(F.col("customer_lifetime_days") > 0, F.col("total_purchases") / (F.col("customer_lifetime_days") / 30.0)).otherwise(F.col("total_purchases")))
    .withColumn("category_diversity", F.col("unique_products") / F.col("total_purchases"))
    .join(dim_cliente.select("sk_cliente", "nome_completo"), on="sk_cliente", how="left")
    .fillna(0)

display(customer_features)

## 3. Model Training

In [ ]:
# Create label: likely to purchase again in next period
# 1 = active customer (purchased recently and multiple times), 0 = at-risk
training_df = customer_features.withColumn(
    "label",
    F.when((F.col("total_purchases") > 1) & (F.col("recency_days") < 60), 1).otherwise(0)
)

# Define feature columns
feature_cols = [c for c in training_df.columns if c not in ("sk_cliente", "nome_completo", "label", "first_purchase_date", "last_purchase_date")]
print(f"Features ({len(feature_cols)}): {feature_cols}")

# Assemble and scale features
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features_raw")
scaler = StandardScaler(inputCol="features_raw", outputCol="features", withStd=True, withMean=False)
preprocessing = Pipeline(stages=[assembler, scaler])

preprocessed = preprocessing.fit(training_df).transform(training_df)
preprocessed = preprocessed.select("features", "label")

# Split data
train_data, test_data = preprocessed.randomSplit([0.8, 0.2], seed=42)

print(f"Train: {train_data.count()}, Test: {test_data.count()}")
print(f"Label distribution in train: {train_data.groupBy('label').count().collect()}")

In [ ]:
# Set MLflow experiment
mlflow.set_experiment("/Shared/retail_sales/purchase_propensity")

# Models to train
models = {
    "logistic_regression": LogisticRegression(featuresCol="features", labelCol="label", maxIter=10),
    "random_forest": RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=10),
    "gbt": GBTClassifier(featuresCol="features", labelCol="label", maxIter=10)
}

auc_evaluator = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")
acc_evaluator = MulticlassClassificationEvaluator(labelCol="label", metricName="accuracy")

# Train and evaluate models
results = {}

with mlflow.start_run(run_name="purchase_propensity_training") as run:
    # Log parameters
    mlflow.log_param("n_features", len(feature_cols))
    mlflow.log_param("feature_names", str(feature_cols[:10]))
    
    for name, model in models.items():
        print(f"Training {name}...")
        trained = model.fit(train_data)
        predictions = trained.transform(test_data)
        
        auc = auc_evaluator.evaluate(predictions)
        acc = acc_evaluator.evaluate(predictions)
        
        results[name] = {"model": trained, "auc": auc, "accuracy": acc}
        
        # Log metrics
        mlflow.log_metric(f"{name}_auc", auc)
        mlflow.log_metric(f"{name}_accuracy", acc)
        
        print(f"{name} - AUC: {auc:.4f}, Accuracy: {acc:.4f}")
    
    # Select and log best model
    best_name = max(results.keys(), key=lambda k: results[k]["auc"])
    best_result = results[best_name]
    
    mlflow.spark.log_model(
        best_result["model"],
        artifact_path="model",
        registered_model_name="purchase_propensity_model"
    )
    
    mlflow.log_metric("best_model_auc", best_result["auc"])
    mlflow.log_param("best_model_name", best_name)
    
    print(f"Best model: {best_name} (AUC={best_result['auc']:.4f})")
    print(f"Run ID: {run.info.run_id}")

display(test_data.limit(20))

## 4. Model Scoring

In [ ]:
# Load the best model from registry
model_uri = f"models:/purchase_propensity_model/latest"
model = mlflow.spark.load_model(model_uri)
print(f"Model loaded from: {model_uri}")

# Score all customers
scored = model.transform(preprocessed)

scored = scored.withColumn(
    "purchase_probability", 
    F.col("probability").getItem(1)
)

scored = scored.join(
    customer_features.select("sk_cliente", "nome_completo"),
    on="sk_cliente",
    how="left"
)

final_scores = scored.select(
    "sk_cliente",
    "nome_completo",
    "label",
    "prediction",
    F.round("purchase_probability", 4).alias("purchase_probability")
).orderBy("purchase_probability", ascending=False)

print(f"Total scored customers: {final_scores.count()}")
display(final_scores)

In [ ]:
# Write scores to Gold table
catalog = "retails_sales_dev"

final_scores.write.format("delta").mode("overwrite").saveAsTable(
    f"{catalog}.gold.customer_purchase_scores"
)

print(f"Scores saved to {catalog}.gold.customer_purchase_scores")